# Sprint 6 — Polymorphism, Inheritance & Blueprint Classes

Spec: [`sprint-6-tasks.md`](../../litemapper/docs/requirements/sprint-6-tasks.md) — tasks S6-T00..T07.

Sprint 6 layers **inheritance-aware dispatch** on top of the Sprint 1–5 scaffolding, introduces the reusable **`MappingBlueprint`** class for sharing configuration across projects, and completes the fluent `IBindingRule` / `IPropertyRule` surface.

| Scope                                        | Task    |
| -------------------------------------------- | ------- |
| Polymorphic dispatch (base → derived DTO)    | S6-T01  |
| `ExtendWith<TDerivedOrigin, TDerivedTarget>` | S6-T02  |
| `Materialize<TConcrete>()` for interfaces    | S6-T03  |
| `InheritFrom<TBaseOrigin, TBaseTarget>()`    | S6-T04  |
| `MappingBlueprint` + `IBlueprintBuilder`     | S6-T05  |
| Full fluent `IBindingRule` chain             | S6-T06  |
| Full fluent `IPropertyRule` chain            | S6-T07  |


## Setup


In [ ]:
#r "../src/SmartMapp.Net/bin/Release/net10.0/SmartMapp.Net.dll"
using SmartMapp.Net;
using SmartMapp.Net.Abstractions;
Console.WriteLine("Ready.");


## 1. `ExtendWith<TDerivedOrigin, TDerivedTarget>()` — polymorphic pair registration

Register the base `Bind<Shape, ShapeDto>` and declare each derived pair with `.ExtendWith<>`. At runtime `sculptor.Map<Shape, ShapeDto>(circle)` returns a `CircleDto` — not a sliced `ShapeDto`.


In [ ]:
public abstract class Shape    { public int Id { get; init; } public string Name { get; init; } = ""; }
public sealed    class Circle    : Shape { public double Radius { get; init; } }
public sealed    class Rectangle : Shape { public double Width { get; init; } public double Height { get; init; } }

public abstract class ShapeDto    { public int Id { get; set; } public string Name { get; set; } = ""; }
public sealed    class CircleDto    : ShapeDto { public double Radius { get; set; } }
public sealed    class RectangleDto : ShapeDto { public double Width { get; set; } public double Height { get; set; } }

var sculptor = new SculptorBuilder()
    .Configure(o =>
    {
        o.Bind<Shape, ShapeDto>(rule => rule
            .ExtendWith<Circle, CircleDto>()
            .ExtendWith<Rectangle, RectangleDto>());
        o.Bind<Circle,    CircleDto>(_ => { });
        o.Bind<Rectangle, RectangleDto>(_ => { });
    })
    .Forge();

Shape circle = new Circle    { Id = 1, Name = "unit",   Radius = 1.0 };
Shape rect   = new Rectangle { Id = 2, Name = "square", Width = 5, Height = 5 };

var circleDto = sculptor.Map<Shape, ShapeDto>(circle);
var rectDto   = sculptor.Map<Shape, ShapeDto>(rect);

Console.WriteLine($"circle runtime type = {circleDto.GetType().Name}  (expected CircleDto)");
Console.WriteLine($"rect   runtime type = {rectDto.GetType().Name}    (expected RectangleDto)");
Console.WriteLine($"(circleDto as CircleDto).Radius = {((CircleDto)circleDto).Radius}");
Console.WriteLine($"(rectDto as RectangleDto).Width = {((RectangleDto)rectDto).Width}");


## 2. `Materialize<TConcrete>()` — concrete target for an interface binding

When the target is an interface or abstract type the mapper needs to know which concrete to construct. `Materialize<TConcrete>()` pins the answer.


In [ ]:
public sealed class Product     { public int Id { get; init; } public string Name { get; init; } = ""; }
public interface   IProductView { int    Id { get; }  string Name { get; } }
public sealed class ProductView : IProductView { public int Id { get; set; } public string Name { get; set; } = ""; }

var sculptor = new SculptorBuilder()
    .Configure(o => o.Bind<Product, IProductView>(rule => rule.Materialize<ProductView>()))
    .Forge();

IProductView view = sculptor.Map<Product, IProductView>(new Product { Id = 42, Name = "Mech Keyboard" });
Console.WriteLine($"Runtime type : {view.GetType().Name}");
Console.WriteLine($"Value        : Id={view.Id}, Name={view.Name}");


## 3. `InheritFrom<TBase, TBaseDto>()` — blueprint inheritance

Let a derived pair inherit the base pair's property rules, then override/extend. Avoids restating common bindings on every derived pair.


In [ ]:
public class Entity        { public int Id { get; init; } public DateTime CreatedAt { get; init; } }
public sealed class Task_       : Entity { public string Title { get; init; } = ""; }

public class EntityDto     { public int Id { get; set; } public string CreatedAtIso { get; set; } = ""; }
public sealed class TaskDto_    : EntityDto { public string Title { get; set; } = ""; }

var sculptor = new SculptorBuilder()
    .Configure(o =>
    {
        o.Bind<Entity, EntityDto>(rule => rule
            .Property(d => d.CreatedAtIso, p => p.From(e => e.CreatedAt.ToString("u"))));
        o.Bind<Task_, TaskDto_>(rule => rule.InheritFrom<Entity, EntityDto>());
    })
    .Forge();

var task = new Task_ { Id = 1, CreatedAt = new DateTime(2026, 4, 22, 9, 0, 0, DateTimeKind.Utc), Title = "ship rc.1" };
var dto = sculptor.Map<Task_, TaskDto_>(task);
Console.WriteLine($"Id           = {dto.Id}");
Console.WriteLine($"CreatedAtIso = {dto.CreatedAtIso}   (inherited from Entity → EntityDto rule)");
Console.WriteLine($"Title        = {dto.Title}");


## 4. `MappingBlueprint` class — reusable, shippable configuration

Pull fluent configuration out of call-site code into a class the host project can ship, version, and unit-test independently.


In [ ]:
public sealed class Order    { public int Id { get; init; } public List<Line> Lines { get; init; } = new(); }
public sealed class Line     { public string Sku { get; init; } = ""; public int Qty { get; init; } public decimal Price { get; init; } }
public sealed class OrderDto { public int Id { get; set; } public decimal Total { get; set; } public List<LineDto> Lines { get; set; } = new(); }
public sealed class LineDto  { public string Sku { get; set; } = ""; public int Qty { get; set; } public decimal Price { get; set; } }

public sealed class OrderBlueprint : MappingBlueprint
{
    public override void Design(IBlueprintBuilder plan)
    {
        plan.Bind<Line, LineDto>();
        plan.Bind<Order, OrderDto>()
            .Property(d => d.Total, p => p.From(o => o.Lines.Sum(l => l.Qty * l.Price)))
            .OnMapped((src, dst) => Console.WriteLine($"  (hook) mapped Order#{src.Id} → total={dst.Total:0.00}"));
    }
}

var sculptor = new SculptorBuilder()
    .UseBlueprint<OrderBlueprint>()
    .Forge();

var order = new Order
{
    Id = 7,
    Lines = { new Line { Sku = "A", Qty = 2, Price = 10m }, new Line { Sku = "B", Qty = 1, Price = 15m } }
};

var dto = sculptor.Map<Order, OrderDto>(order);
Console.WriteLine($"Result: Id={dto.Id}, Total={dto.Total:0.00}, Lines={dto.Lines.Count}");


## 5. Full fluent chain — `OnMapping` + `OnMapped` + `BuildWith` + `FallbackTo`

Every knob on `IBindingRule` composes. Below: a factory, pre-map + post-map hooks, and a fallback default for null origin.


In [ ]:
var sculptor = new SculptorBuilder()
    .Configure(o => o.Bind<Line, LineDto>(rule => rule
        .BuildWith(l => new LineDto { Sku = l.Sku.ToUpperInvariant() })
        .OnMapping((src, dst) => Console.WriteLine($"  OnMapping: src.Sku = {src.Sku}"))
        .OnMapped ((src, dst) => Console.WriteLine($"  OnMapped : dst.Sku = {dst.Sku}, Qty = {dst.Qty}"))))
    .Forge();

var dto = sculptor.Map<Line, LineDto>(new Line { Sku = "abc", Qty = 4, Price = 9.99m });
Console.WriteLine($"Final: Sku={dto.Sku}, Qty={dto.Qty}, Price={dto.Price:0.00}");


## Next

- **`sprint-07-attributes-validation-diagnostics.ipynb`** — attribute-driven pairs, `Validate()`, `Inspect<S,D>()`, `MappingAtlas` + DOT export.
